[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/measurement.ipynb)

# Units, measurement and dimensions full course: the practice set, in Python

Twenty-eight exercises in three tiers, every answer checked: **tier 1** warm-ups (one idea, one
step), **tier 2** core problems (conversions, significant figures in calculations, uncertainty in sums, products and
powers, dimensional formulas and dimensional analysis) and **tier 3** stretch problems (combine ideas, spot the trap,
run a formula backwards). Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the course's rules written as small functions
(`decimal.Decimal` for rounding, tuples of powers for dimensions), so you can use them on your own measurements. And
a few cells you can move yourself: a pendulum's length and its swing, and a mass on a spring, each simulated step by
step and set beside its formula.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Decimals as
decimals; a percentage as a plain number, `2.0` for 2.0 % (typing the % sign is fine too); a unit after the number is
fine (`15 m/s`). Powers of M, L and T as whole numbers, `-2`; a half as `0.5` or `1/2`. Numbers without thousands
separators: `15000`, not `15,000`. You can type the arithmetic too: `54*1000/3600`. For several answers, separate
them with commas, in the order the question lists them (the box's label shows the order). You can also put your
answer between the quotes in a cell's last line, `practice("P1.1", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every rule and formula from the course on two pages:
[measurement cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/measurement_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, textwrap
from decimal import Decimal, ROUND_HALF_UP, ROUND_HALF_EVEN
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # bodies and data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# the physics courses' colours: velocity SKY · acceleration MINT · force CORAL · energy VIOLET · bodies and
# displacement CHALK · a formula's prediction DIM, dashed · the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets, pi and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id == "pi": return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(m/s²|m/s\*\*2|m/s2|m/s|km/h²|km/h\*\*2|km/h|cm³|cm\*\*3|cm²|cm\*\*2|m²|m³|µm|um|mm|cm|km|ms|kg|"
        r"g·cm/s²|g cm/s2|lbf·s|lbf s|lbf|n·s|n s|j|n|m|s|g|%|swings?|sig figs?|significant figures?)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("π", "pi").replace("√", "sqrt")
    for t in (s, re.sub(UNIT, "", s, flags=re.I).strip()):
        try: return calc(t)
        except Exception: pass
    return " ".join(s.lower().split())

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own questions, answers and working ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Units and the SI',
     "q": 'A scooter rides at 54 km/h. What is that in m/s?',
     "parts": ['speed (m/s)'],
     "answer": [15.0], "tol": [0.005],
     "working": '54 × 1000/3600 = 15 m/s (multiply by one: 1000 m/1 km and 1 h/3600 s)'},
    {"id": 'P1.2', "tier": 1, "group": 'Units and the SI',
     "q": 'Write 0.00072 m in micrometres (µm).',
     "parts": ['length (µm)'],
     "answer": [720.0], "tol": [0.005],
     "working": '1 m = 10⁶ µm: 0.00072 × 10⁶ = 720 µm'},
    {"id": 'P1.3', "tier": 1, "group": 'Significant figures',
     "q": 'How many significant figures has 0.05080 kg?',
     "parts": ['significant figures'],
     "answer": [4], "tol": [0],
     "working": 'Leading zeros never count; the zero between 5 and 8 counts; the trailing zero after the point counts: 5, 0, 8, 0 → 4'},
    {"id": 'P1.4', "tier": 1, "group": 'Significant figures',
     "q": 'Round 6.0349 to three significant figures (round half up).',
     "parts": ['rounded'],
     "answer": [6.03], "tol": [0.005],
     "working": 'The first dropped digit is 4 (< 5): 6.03'},
    {"id": 'P1.5', "tier": 1, "group": 'Significant figures',
     "q": 'Add 12.4 m and 3.861 m, keeping the right number of decimal places.',
     "parts": ['the sum (m)'],
     "answer": [16.3], "tol": [0.005],
     "working": '16.261 → one decimal place, as in 12.4: 16.3 m'},
    {"id": 'P1.6', "tier": 1, "group": 'Uncertainty in a result',
     "q": 'A length is 25.0 ± 0.5 cm. What is its percentage uncertainty?',
     "parts": ['percentage uncertainty (%)'],
     "answer": [2.0], "tol": [0.005],
     "working": '0.5/25.0 = 0.02 = 2.0 %'},
    {"id": 'P1.7', "tier": 1, "group": 'Uncertainty in a result',
     "q": 'Add 4.6 ± 0.1 cm and 2.3 ± 0.1 cm. What is the uncertainty of the sum?',
     "parts": ['uncertainty of the sum (cm)'],
     "answer": [0.2], "tol": [0.005],
     "working": 'In a sum, absolute uncertainties add: 0.1 + 0.1 = 0.2 cm (6.9 ± 0.2 cm)'},
    {"id": 'P1.8', "tier": 1, "group": 'Dimensions',
     "q": 'Give the powers of L and T in the dimensions of acceleration.',
     "parts": ['power of L', 'power of T'],
     "answer": [1, -2], "tol": [0, 0],
     "working": 'Acceleration = speed ÷ time = L T⁻¹ ÷ T = L T⁻²: (1, −2)'},
    {"id": 'P1.9', "tier": 1, "group": 'Dimensions',
     "q": 'Give the powers of M, L and T in the dimensions of density.',
     "parts": ['power of M', 'power of L', 'power of T'],
     "answer": [1, -3, 0], "tol": [0, 0, 0],
     "working": 'Density = mass ÷ volume = M ÷ L³ = M L⁻³ T⁰: (1, −3, 0)'},
    {"id": 'P1.10', "tier": 1, "group": 'Dimensional analysis',
     "q": 'Is v = u + at dimensionally consistent? Type 1 for yes, 0 for no.',
     "parts": ['1 (yes) or 0 (no)'],
     "answer": [1], "tol": [0],
     "working": 'v and u: L T⁻¹; at: L T⁻² × T = L T⁻¹. Every term matches: 1'},
    {"id": 'P2.1', "tier": 2, "group": 'Units and the SI',
     "q": 'A desk top has an area of 1.5 m². What is that in cm²?',
     "parts": ['area (cm²)'],
     "answer": [15000.0], "tol": [0.005],
     "working": '1 m = 100 cm, so 1 m² = 100² cm² = 10,000 cm²: 1.5 × 10,000 = 15,000 cm²'},
    {"id": 'P2.2', "tier": 2, "group": 'Significant figures',
     "q": 'A rug is 3.42 m by 2.1 m. What is its area, to the right significant figures?',
     "parts": ['area (m²)'],
     "answer": [7.2], "tol": [0.005],
     "working": '3.42 × 2.1 = 7.182 → two significant figures (2.1): 7.2 m²'},
    {"id": 'P2.3', "tier": 2, "group": 'Significant figures',
     "q": 'A ball rolls 25.0 m in 3.27 s. What is its average speed, to the right significant figures?',
     "parts": ['speed (m/s)'],
     "answer": [7.65], "tol": [0.005],
     "working": '25.0/3.27 = 7.64526 → three significant figures: 7.65 m/s'},
    {"id": 'P2.4', "tier": 2, "group": 'Uncertainty in a result',
     "q": 'Subtract: (45.6 ± 0.2) cm − (12.3 ± 0.1) cm. Give the result and its uncertainty.',
     "parts": ['the difference (cm)', 'its ± (cm)'],
     "answer": [33.3, 0.30000000000000004], "tol": [0.005, 0.005],
     "working": '33.3 cm; absolute uncertainties add even in a difference: 0.2 + 0.1 = 0.3 → 33.3 ± 0.3 cm'},
    {"id": 'P2.5', "tier": 2, "group": 'Uncertainty in a result',
     "q": 'A card is (12.0 ± 0.1) cm by (5.0 ± 0.1) cm. Give its area and the uncertainty, quoted.',
     "parts": ['area (cm²), quoted', 'its ± (cm²)'],
     "answer": [60.0, 2.0], "tol": [0.005, 0.005],
     "working": 'Area 60.0; relative 0.1/12.0 + 0.1/5.0 = 2.83 % → ±1.7 → one significant figure: ±2 → 60 ± 2 cm²'},
    {"id": 'P2.6', "tier": 2, "group": 'Uncertainty in a result',
     "q": "A cube's side is 4.00 ± 0.02 cm. Give its volume and the uncertainty, quoted.",
     "parts": ['volume (cm³), quoted', 'its ± (cm³)'],
     "answer": [64.0, 1.0], "tol": [0.005, 0.005],
     "working": '64.0 cm³; the power multiplies: 3 × 0.5 % = 1.5 % → ±0.96 → ±1 → 64 ± 1 cm³'},
    {"id": 'P2.7', "tier": 2, "group": 'Dimensions',
     "q": 'Pressure is force per area. Give the powers of M, L and T.',
     "parts": ['power of M', 'power of L', 'power of T'],
     "answer": [1, -1, -2], "tol": [0, 0, 0],
     "working": 'M L T⁻² ÷ L² = M L⁻¹ T⁻²: (1, −1, −2)'},
    {"id": 'P2.8', "tier": 2, "group": 'Dimensional analysis',
     "q": 'How many g·cm/s² make one newton?',
     "parts": ['g·cm/s² in one newton'],
     "answer": [100000.0], "tol": [0.005],
     "working": '1 N = 1 kg·m/s² = 1000 g × 100 cm/s² = 100,000 g·cm/s²'},
    {"id": 'P2.9', "tier": 2, "group": 'Dimensional analysis',
     "q": "A spring's period T depends on the mass m and the spring constant k (N/m). Find the powers x and y in T = c mˣ kʸ.",
     "parts": ['x', 'y'],
     "answer": [0.5, -0.5], "tol": [0, 0],
     "working": 'T¹ = Mˣ (M T⁻²)ʸ: M: x + y = 0; T: −2y = 1 → y = −½, x = ½: T = c √(m/k)'},
    {"id": 'P2.10', "tier": 2, "group": 'Dimensional analysis',
     "q": 'What pendulum length gives a period of 2.00 s, with g = 9.8 m/s²? Give metres to three decimals.',
     "parts": ['length (m, three decimals)'],
     "answer": [0.993], "tol": [0.0005],
     "working": 'l = g T²/(4π²) = 9.8 × 4/39.478 = 0.9929 m'},
    {"id": 'P3.1', "tier": 3, "group": 'Significant figures',
     "q": 'Round 3.4449 to two significant figures, once. Then see what rounding in steps (to 4, 3, then 2) gives.',
     "parts": ['rounded once', 'rounded in steps'],
     "answer": [3.4, 3.5], "tol": [0.005, 0.005],
     "working": 'Once: the first dropped digit is 4 → 3.4. In steps: 3.445 → 3.45 → 3.5, wrong. Round once, at the end'},
    {"id": 'P3.2', "tier": 3, "group": 'Significant figures',
     "q": 'Two stopwatch times, 25.36 s and 25.31 s, are subtracted. How many significant figures has the difference?',
     "parts": ['significant figures'],
     "answer": [1], "tol": [0],
     "working": '25.36 − 25.31 = 0.05 s: one significant figure, though each time had four'},
    {"id": 'P3.3', "tier": 3, "group": 'Uncertainty in a result',
     "q": 'A football (m = 0.450 ± 0.005 kg) is kicked at v = 20.0 ± 0.4 m/s. Give its kinetic energy ½mv² and the uncertainty, quoted.',
     "parts": ['kinetic energy (J), quoted', 'its ± (J)'],
     "answer": [90.0, 5.0], "tol": [0.005, 0.005],
     "working": '½ × 0.450 × 400 = 90.0 J; relative 1.11 % + 2 × 2 % = 5.11 % → ±4.6 → 90 ± 5 J (the ½ is exact)'},
    {"id": 'P3.4', "tier": 3, "group": 'Uncertainty in a result',
     "q": 'A 0.500 m pendulum is timed by hand (±0.2 s on any timing). How many swings must you time for the period to be within 0.5 %?',
     "parts": ['swings'],
     "answer": [29], "tol": [0],
     "working": 'T = 1.419 s. Timing N swings: 0.2/(N × 1.419) ≤ 0.005 → N ≥ 28.2 → 29 swings'},
    {"id": 'P3.5', "tier": 3, "group": 'Dimensions',
     "q": "Newton's law of gravitation is F = G m₁ m₂ / r². Run it backwards: give the powers of M, L and T in G.",
     "parts": ['power of M', 'power of L', 'power of T'],
     "answer": [-1, 3, -2], "tol": [0, 0, 0],
     "working": 'G = F r²/(m₁ m₂) = M L T⁻² × L² ÷ M² = M⁻¹ L³ T⁻²: (−1, 3, −2)'},
    {"id": 'P3.6', "tier": 3, "group": 'Dimensional analysis',
     "q": 'Convert g = 9.8 m/s² into km/h² (give the unrounded number, then think about significant figures).',
     "parts": ['g (km/h², unrounded)'],
     "answer": [127008.00000000001], "tol": [0.005],
     "working": '9.8 × (1 km/1000 m) × (3600 s/1 h)² = 127,008 km/h², quoted as 1.3 × 10⁵ km/h²'},
    {"id": 'P3.7', "tier": 3, "group": 'Dimensional analysis',
     "q": 'A 1.200 ± 0.002 m pendulum makes 10 swings in 22.0 ± 0.2 s. Find g and its uncertainty, quoted.',
     "parts": ['g (m/s²), quoted', 'its ± (m/s²)'],
     "answer": [9.8, 0.2], "tol": [0.005, 0.005],
     "working": 'T = 2.20 ± 0.02 s; g = 4π² × 1.200/2.20² = 9.788; relative 0.17 % + 2 × 0.91 % = 1.98 % → 9.8 ± 0.2 m/s²'},
    {"id": 'P3.8', "tier": 3, "group": 'Dimensional analysis',
     "q": 'A thruster gives an impulse of 25 N·s. What is that in pound-force seconds?',
     "parts": ['impulse (lbf·s)'],
     "answer": [5.620223577492762], "tol": [0.005],
     "working": '1 lbf = 0.45359237 kg × 9.80665 m/s² = 4.4482 N, so 25/4.4482 = 5.62 lbf·s'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    got = [float(g) for g in got]
    return len(got) == len(answers(k)) and all(right(g, w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't, units and arithmetic typed either way
assert right(read_part("54*1000/3600"), 15, 0.005) and right(read_part("15 m/s"), 15, 0.005)
assert not right(read_part("14"), 15, 0.005) and right(read_part("2.0 %"), 2.0, 0.005) and right(read_part("1/2"), 0.5, 0)
assert [read_part(p) for p in "1, −2".split(",")] == [1, -2]

## Tier 1 · Warm-up

One idea, one step: a speed in metres per second, a length in micrometres, counting significant figures, rounding,
adding with the right decimal places, a percentage uncertainty, the uncertainty of a sum, the dimensions of
acceleration and of density, and checking an equation of motion.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

**Units**: multiply by one. 1000 m over 1 km is one, and so is 1 h over 3600 s, so 54 km/h × 1000/3600 = 15 m/s.

**Significant figures and rounding**: a computer stores a number, not the way you wrote it, so `0.05080` and
`0.0508` are the same float. To count significant figures, keep the number as text. To round the way the course
does (round half up, on the written decimal digits), use `decimal.Decimal` with `ROUND_HALF_UP`: it rounds the digits
you wrote, not a binary approximation of them. The functions below are the course's rules, word for word.

**Dimensions**: write each quantity's dimensions as a tuple of powers (M, L, T). Multiplying quantities adds the
powers; dividing subtracts them; a square doubles them.

In [ ]:
def sig_figs(s):
    """Significant figures of a number AS WRITTEN: "0.003040" → 4, "4500" → 2, "4.50e3" (4.50 × 10³) → 3."""
    mant = s.lower().split("e")[0].lstrip("+-")
    digits = mant.replace(".", "").lstrip("0")                  # leading zeros never count
    if "." in mant: return len(digits)                           # zeros after a point count
    return len(digits.rstrip("0"))                               # a whole number's trailing zeros don't

def round_sf(x, n, mode=ROUND_HALF_UP):
    """x (written as text, or a Decimal) to n significant figures, keeping its trailing zeros."""
    d = Decimal(str(x))
    return d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=mode)

def round_dp(x, n, mode=ROUND_HALF_UP):
    """x to n decimal places."""
    return Decimal(str(x)).quantize(Decimal(1).scaleb(-n), rounding=mode)

def quote(value, unc):
    """(value, uncertainty) as quoted: the uncertainty to one significant figure, the value to the same place."""
    u = round_sf(round_sf(unc, 1), 1)   # twice: a carry (0.96 → 1.0) moves the place
    return Decimal(str(value)).quantize(Decimal(1).scaleb(u.as_tuple().exponent), rounding=ROUND_HALF_UP), u

def rel_product(*terms):
    """Z = Aᵖ Bᑫ …: ΔZ/Z = |p| ΔA/A + |q| ΔB/B + …; each term is (value, uncertainty, power)."""
    return sum(abs(p) * du / u for u, du, p in terms)

DIMS = {"length": (0, 1, 0), "mass": (1, 0, 0), "time": (0, 0, 1), "area": (0, 2, 0), "volume": (0, 3, 0),
        "speed": (0, 1, -1), "acceleration": (0, 1, -2), "force": (1, 1, -2), "work": (1, 2, -2),
        "power": (1, 2, -3), "pressure": (1, -1, -2), "density": (1, -3, 0), "angle": (0, 0, 0)}
def dmul(*ds): return tuple(sum(c) for c in zip(*ds))           # multiply quantities: add the powers
def dpow(d, n): return tuple(n * c for c in d)                  # a power: multiply the powers

print(sig_figs("0.003040"), sig_figs("4500"), sig_figs("4.50e3"), "  ", round_sf("0.003040", 2), round_sf("1.996", 3))
assert [sig_figs(s) for s in ("14.32", "2.05", "1005", "0.0045", "2.50", "4500", "4.50e3")] == [4, 3, 4, 2, 3, 2, 3]
assert str(round_sf("0.003040", 2)) == "0.0030" and str(round_sf("1.996", 3)) == "2.00"     # the zeros stay

KMH = 1000 / 3600                                                     # one km/h in m/s
mine = {
    "P1.1": [54 * KMH],
    "P1.2": [0.00072 * 10 ** 6],
    "P1.3": [sig_figs("0.05080")],
    "P1.4": [round_sf("6.0349", 3)],
    "P1.5": [round_dp(Decimal("12.4") + Decimal("3.861"), 1)],         # fewest decimal places: one
    "P1.6": [100 * 0.5 / 25.0],
    "P1.7": [0.1 + 0.1],                                              # in a sum, absolute uncertainties add
    "P1.8": DIMS["acceleration"][1:],
    "P1.9": DIMS["density"],
    "P1.10": [int(DIMS["speed"] == dmul(DIMS["acceleration"], DIMS["time"]))],   # v = u + at: [at] = [v]?
}
for k, v in mine.items():
    print(f"{k}: {', '.join(str(x) for x in v)}")
    assert agrees(k, v), k

### Why Python's `round()` can disagree on an exact five

The course rounds half up: 2.745 to three significant figures is 2.75. Python's built-in `round()` does two
different things. On a number stored exactly, a five rounds to the **even** neighbour ("half to even", the rule some
textbooks and computers use): `round(2.5)` is 2 and `round(0.125, 2)` is 0.12. And most decimals aren't stored
exactly: 2.675 is really 2.67499999…, so `round(2.675, 2)` gives 2.67. `Decimal` works on the digits you wrote.

In [ ]:
print("round(2.5) =", round(2.5), "  round(3.5) =", round(3.5), "  round(0.125, 2) =", round(0.125, 2))
print("2.675 is stored as", Decimal(2.675), " so round(2.675, 2) =", round(2.675, 2))
print("Decimal, half up:  ", round_dp("0.125", 2), " ", round_dp("2.675", 2), " ", round_sf("2.745", 3))
print("Decimal, half even:", round_dp("0.125", 2, ROUND_HALF_EVEN), " ", round_sf("2.745", 3, ROUND_HALF_EVEN))
assert round(2.5) == 2 and round(3.5) == 4 and round(0.125, 2) == 0.12 and round(2.675, 2) == 2.67
assert str(round_dp("0.125", 2)) == "0.13" and str(round_dp("2.675", 2)) == "2.68"
assert str(round_sf("2.745", 3)) == "2.75" and str(round_sf("2.745", 3, ROUND_HALF_EVEN)) == "2.74"
# round once, at the end: 2.4449 to two figures is 2.4; in steps (4, then 3, then 2) it drifts to 2.5
steps = round_sf(round_sf(round_sf("2.4449", 4), 3), 2)
print("2.4449 → once:", round_sf("2.4449", 2), "  in steps:", steps)
assert str(round_sf("2.4449", 2)) == "2.4" and str(steps) == "2.5"

### A falling ball: five stopwatch readings and the simulation

A ball is dropped from 2.00 m and timed by hand five times. The readings scatter: quote the mean, give or take half
the range, 0.64 ± 0.02 s. The cell below also **simulates** the fall: it steps the ball forward a thousandth of a
second at a time with the fourth-order Runge–Kutta method (a careful way of following a motion in small steps, from
its acceleration) and stops when it reaches the floor. The simulated time, 0.639 s, is the formula's √(2h/g), and it
sits inside the readings' band.

In [ ]:
G = 9.8                                                                # m/s², the course's g

def rk4(deriv, state, dt, stop, t_max):
    """Step a motion forward with fourth-order Runge–Kutta until stop(state) is true. Returns the times and states."""
    t, s = 0.0, np.array(state, float)
    ts, ss = [t], [s]
    while t < t_max:
        k1 = deriv(t, s); k2 = deriv(t + dt / 2, s + dt / 2 * k1)
        k3 = deriv(t + dt / 2, s + dt / 2 * k2); k4 = deriv(t + dt, s + dt * k3)
        s = s + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4); t += dt
        ts.append(t); ss.append(s)
        if stop(s): break
    return np.array(ts), np.array(ss)

def crossing(ts, ss, k=0):
    """The time at which component k of the state last crossed zero (between the last two steps, by a straight line)."""
    (t0, t1), (a, b) = ts[-2:], ss[-2:, k]
    return t0 + (t1 - t0) * a / (a - b)

h = 2.00
ts, ss = rk4(lambda t, s: np.array([s[1], -G]), [h, 0.0], 1e-3, lambda s: s[0] < 0, 5)
t_fall = crossing(ts, ss)
drop = [0.62, 0.65, 0.65, 0.66, 0.62]                                 # illustrative readings · our own data
mean, half = np.mean(drop), (max(drop) - min(drop)) / 2
print(f"readings {drop}: {mean:.2f} ± {half:.2f} s;  simulated {t_fall:.4f} s;  √(2h/g) = {math.sqrt(2 * h / G):.4f} s")
assert abs(t_fall - math.sqrt(2 * h / G)) < 1e-6 and round(t_fall, 3) == 0.639
assert round(mean, 9) == 0.64 and round(half, 9) == 0.02 and abs(t_fall - mean) < half

fig, ax = plt.subplots(figsize=(11.5, 3.6))
fig.subplots_adjust(left=0.08, right=0.98, top=0.82, bottom=0.18)
ax.axhspan(mean - half, mean + half, color=DIM, alpha=0.18, lw=0)
ax.axhline(mean, color=DIM, ls="--", lw=1.6)
ax.scatter(range(1, 6), drop, s=70, color=CHALK, zorder=3)
ax.axhline(t_fall, color=GOLD, lw=2.4)
ax.set_xticks(range(1, 6)); ax.set_xlabel("reading"); ax.set_ylabel("time (s)"); ax.set_ylim(0.60, 0.68)
fig.suptitle(f"five readings, {mean:.2f} ± {half:.2f} s; the simulated fall, {t_fall:.3f} s, sits inside", x=0.08,
             ha="left", fontsize=15, color=GOLD)
footnote(fig, "illustrative readings · our own data · simulated"); plt.show()

## Tier 2 · Core

Several steps each: an area in square centimetres, significant figures in a product and a quotient, the uncertainty
of a difference, of a product and of a power, the dimensions of pressure, a newton in grams and centimetres, a law
from its dimensions, and a pendulum built for a given period.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Calculations**: × and ÷ keep the fewest significant figures of the inputs; + and − keep the fewest decimal places.
**Uncertainty**: in a sum or a difference, absolute uncertainties add; in a product or a quotient, relative
uncertainties add; a power multiplies the relative uncertainty. Then `quote()`: the uncertainty to one significant
figure, the value to the same place. **Dimensional analysis**: matching the powers of M, L and T on both sides gives a
few equations in the unknown powers; `numpy.linalg.solve` solves them.

In [ ]:
mine = {}
mine["P2.1"] = [1.5 * 100 ** 2]                                        # 1 m² = 100² cm²
mine["P2.2"] = [round_sf(Decimal("3.42") * Decimal("2.1"), 2)]         # 2.1 has two significant figures
mine["P2.3"] = [round_sf(Decimal("25.0") / Decimal("3.27"), 3)]
mine["P2.4"] = [45.6 - 12.3, 0.2 + 0.1]                                # a difference: absolute uncertainties add
area = 12.0 * 5.0
mine["P2.5"] = quote(area, area * rel_product((12.0, 0.1, 1), (5.0, 0.1, 1)))
vol = 4.00 ** 3
mine["P2.6"] = quote(vol, vol * rel_product((4.00, 0.02, 3)))          # the power 3 triples the relative uncertainty
mine["P2.7"] = dmul(DIMS["force"], dpow(DIMS["area"], -1))             # pressure = force ÷ area
mine["P2.8"] = [1000 * 100]                                            # 1 kg = 1000 g, 1 m = 100 cm
k_dims = dmul(DIMS["force"], dpow(DIMS["length"], -1))                 # a spring constant, N/m: M T⁻²
# T = c mˣ kʸ: match the powers of M and of T (L doesn't appear): M: x + y = 0; T: −2y = 1
x, y = np.linalg.solve([[1, k_dims[0]], [0, k_dims[2]]], [0, 1])
mine["P2.9"] = [x, y]
mine["P2.10"] = [round(G * 2.00 ** 2 / (4 * math.pi ** 2), 3)]         # l = g T² ÷ 4π²
for k, v in mine.items():
    print(f"{k}: {', '.join(str(z) for z in v)}")
    assert agrees(k, v), k
assert [str(z) for z in mine["P2.5"]] == ["60", "2"] and k_dims == (1, 0, -2)

### The pendulum: change its length and its swing

Dimensional analysis says a pendulum's period is T = k √(l/g), and can't say what k is. The simulation finds it: it
swings a pendulum step by step from its real equation of motion (no small-angle shortcut), and times a swing. The
simulated period comes out as 2π √(l/g), so k = 2π. Slide the **length**: a quarter of the length gives half the
period. Slide the **swing** (how far it's pulled back): small swings match the formula; at 60° the real period is
about 7 % longer. The mass of the bob never enters: it cancels out of the equation.

In [ ]:
def pendulum(l, swing_deg, g=G, dt=1e-3):
    """Simulate a pendulum released at rest at swing_deg; return its period (four times the quarter swing)."""
    deriv = lambda t, s: np.array([s[1], -g / l * math.sin(s[0])])   # the mass cancels: m g sinθ ÷ m l
    ts, ss = rk4(deriv, [math.radians(swing_deg), 0.0], dt, lambda s: s[0] < 0, 10)
    return 4 * crossing(ts, ss)

def exact_period(l, swing_deg, g=G):
    """The exact period at any swing: 4 √(l/g) K(sin(θ₀/2)), K by the arithmetic–geometric mean."""
    k = math.sin(math.radians(swing_deg) / 2); a, b = 1.0, math.sqrt(1 - k * k)
    for _ in range(40): a, b = (a + b) / 2, math.sqrt(a * b)
    return 4 * math.sqrt(l / g) * math.pi / (2 * a)

def pendulum_picture(length=1.00, swing=10.0):
    period = pendulum(length, swing)
    formula = 2 * math.pi * math.sqrt(length / G)
    deriv = lambda t, s: np.array([s[1], -G / length * math.sin(s[0])])
    ts, ss = rk4(deriv, [math.radians(swing), 0.0], 2e-3, lambda s: False, 2 * formula)
    fig, ax = plt.subplots(figsize=(11.5, 4.4))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.15)
    ax.plot(ts, np.degrees(ss[:, 0]), color=CHALK, label="simulated swing")
    ax.plot(ts, swing * np.cos(2 * math.pi * ts / formula), color=DIM, ls="--", lw=1.6,
            label="the formula's swing, period 2π√(l/g)")
    ax.axvline(period, color=GOLD, lw=2)
    ax.set_xlabel("time (s)"); ax.set_ylabel("angle (°)"); ax.set_ylim(-1.3 * swing, 1.45 * swing)
    ax.legend(loc="upper right", frameon=False, ncol=2)
    fig.suptitle(f"l = {length:.2f} m, swing {swing:g}°: simulated period {period:.3f} s · 2π√(l/g) = {formula:.3f} s "
                 f"({100 * (period / formula - 1):+.1f} %)", x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig, "simulated · computed"); plt.show()

T1 = pendulum(1.00, 1)                                                # the video's numbers
print(f"1.00 m: {T1:.4f} s, k = T ÷ √(l/g) = {T1 / math.sqrt(1 / G):.4f} (2π = {2 * math.pi:.4f});  0.25 m: "
      f"{pendulum(0.25, 1):.4f} s;  60°: {pendulum(1.00, 60) / T1:.3f} × as long")
assert round(T1, 3) == 2.007 and round(T1 / math.sqrt(1 / G), 3) == 6.283 and round(pendulum(0.25, 1), 3) == 1.004
assert abs(pendulum(1.00, 60) - exact_period(1.00, 60)) < 1e-6 and round(pendulum(1.00, 60) / T1, 3) == 1.073

In [ ]:
interact(pendulum_picture, length=FloatSlider(value=1.00, min=0.10, max=2.00, step=0.05, description="length (m)"),
         swing=FloatSlider(value=10, min=1, max=90, step=1, description="swing (°)"));

### A mass on a spring

The same check for a mass on a spring: dimensions gave T = c √(m/k); the simulation, stepping x'' = −(k/m) x, finds
c = 2π again. A 0.50 kg mass on a 20 N/m spring swings with a period of 0.993 s. Slide the mass and the stiffness: a
heavier mass swings more slowly, a stiffer spring more quickly, and how far you pull it doesn't matter.

In [ ]:
def spring_period(m, k, pull=0.10, dt=1e-3):
    ts, ss = rk4(lambda t, s: np.array([s[1], -k / m * s[0]]), [pull, 0.0], dt, lambda s: s[0] < 0, 20)
    return 4 * crossing(ts, ss)

def spring_picture(mass=0.50, stiffness=20.0):
    period, formula = spring_period(mass, stiffness), 2 * math.pi * math.sqrt(mass / stiffness)
    ts, ss = rk4(lambda t, s: np.array([s[1], -stiffness / mass * s[0]]), [0.10, 0.0], 2e-3, lambda s: False, 2 * formula)
    fig, ax = plt.subplots(figsize=(11.5, 4.0))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.16)
    ax.plot(ts, 100 * ss[:, 0], color=CHALK, label="simulated")
    ax.plot(ts, 10 * np.cos(2 * math.pi * ts / formula), color=DIM, ls="--", lw=1.6, label="2π√(m/k)")
    ax.axvline(period, color=GOLD, lw=2)
    ax.set_xlabel("time (s)"); ax.set_ylabel("stretch (cm)"); ax.set_ylim(-13, 15)
    ax.legend(loc="upper right", frameon=False, ncol=2)
    fig.suptitle(f"m = {mass:.2f} kg, k = {stiffness:g} N/m: simulated period {period:.3f} s · 2π√(m/k) = {formula:.3f} s",
                 x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig, "simulated · computed"); plt.show()

Ts = spring_period(0.50, 20.0)
print(f"0.50 kg on 20 N/m: {Ts:.4f} s;  c = T ÷ √(m/k) = {Ts / math.sqrt(0.50 / 20.0):.4f}")
assert round(Ts, 3) == 0.993 and round(Ts / math.sqrt(0.50 / 20.0), 3) == 6.283

In [ ]:
interact(spring_picture, mass=FloatSlider(value=0.50, min=0.10, max=2.00, step=0.05, description="mass (kg)"),
         stiffness=FloatSlider(value=20, min=5, max=80, step=1, description="k (N/m)"));

## Tier 3 · Stretch

Combine ideas, spot the trap, run a formula backwards: rounding once against rounding in steps, a difference that
loses its figures, the uncertainty of a kinetic energy, how many swings to time, the dimensions of G, g in km/h², g
from a pendulum with its uncertainty, and an impulse in pound-force seconds.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

The same few functions, used together. Running a formula backwards is algebra first (l = gT²/4π², G = F r²/m₁m₂),
then one line of Python. And the course's pendulum measurement of g, from five timings of 20 swings: the mean and
half the range, the period, g = 4π² l/T², and its uncertainty (the power 2 on T doubles T's relative uncertainty).

In [ ]:
mine = {}
mine["P3.1"] = [round_sf("3.4449", 2), round_sf(round_sf(round_sf("3.4449", 4), 3), 2)]
mine["P3.2"] = [sig_figs(str(Decimal("25.36") - Decimal("25.31")))]   # 0.05: one significant figure
ke = 0.5 * 0.450 * 20.0 ** 2                                            # the ½ is exact
mine["P3.3"] = quote(ke, ke * rel_product((0.450, 0.005, 1), (20.0, 0.4, 2)))
T05 = 2 * math.pi * math.sqrt(0.500 / G)
mine["P3.4"] = [math.ceil(0.2 / (0.005 * T05))]                          # 0.2/(N T) ≤ 0.005
mine["P3.5"] = dmul(DIMS["force"], DIMS["area"], dpow(DIMS["mass"], -2))   # G = F r² ÷ (m₁ m₂)
mine["P3.6"] = [G * (1 / 1000) * 3600 ** 2]
g37 = 4 * math.pi ** 2 * 1.200 / 2.20 ** 2                             # T = 22.0 ÷ 10 = 2.20 ± 0.02 s
mine["P3.7"] = quote(g37, g37 * rel_product((1.200, 0.002, 1), (2.20, 0.02, -2)))
LBF = 0.45359237 * 9.80665                                              # one pound-force in newtons, exact
mine["P3.8"] = [25 / LBF]
for k, v in mine.items():
    print(f"{k}: {', '.join(str(z) for z in v)}")
    assert agrees(k, v), k

l, dl = 0.800, 0.001                                                    # the course's pendulum
timings = [35.9, 35.8, 36.0, 36.1, 35.7]                                # five timings of 20 swings · our own data
mean, half = np.mean(timings), (max(timings) - min(timings)) / 2
T, dT = mean / 20, half / 20
g = 4 * math.pi ** 2 * l / T ** 2
rel = rel_product((l, dl, 1), (T, dT, -2))
gq, uq = quote(g, g * rel)
print(f"\nthe pendulum: {mean:.1f} ± {half:.1f} s for 20 swings → T = {T:.3f} ± {dT:.3f} s → g = {g:.3f} m/s², "
      f"± {100 * rel:.2f} % → g = {gq} ± {uq} m/s²")
assert round(T, 6) == 1.795 and round(g, 3) == 9.802 and round(100 * rel, 2) == 1.24 and (str(gq), str(uq)) == ("9.8", "0.1")

## Sources
Every exercise and data set here is our own. The SI's seven base units, each
defined since 20 May 2019 by fixing a constant of nature: BIPM, "Measurement units" (bipm.org, retrieved 2026-11-11).
The international pound is exactly 0.45359237 kg and standard gravity exactly 9.80665 m/s², so one pound-force is
4.448 222 N (NIST Special Publication 811, Appendix B, retrieved 2026-11-11). The order of topics follows a standard
first course in physics (units and measurement).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/ph01_measurement/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("ph01_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"]) == (_e["q"], _e["working"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
        assert all(abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX) and _P.G == G and _P.DIMS == DIMS
    assert _P.DROP == drop and _P.PEND_TIMES == timings and _P.Q10["l"] == (l, dl)
    print("matches the course's practice set")